# Understanding `retriever.py`, step by step

This notebook takes apart the **"R" in RAG**: `cc_copilot/rag/retriever.py` finds the company policy sections that
are relevant to a customer conversation, so the policy agent can judge the agent's reply against **real rules**
instead of guessing.

It also covers `cc_copilot/rag/arabic_text.py`, because the retriever can only match words after they've been
cleaned up by it.

Run the cells in order (Shift + Enter), and change the example questions as you go.

**The big picture.**

```
policy files (.md) ──► 1. split into sections, one per "## REF-1 ..." heading     (load_policy_chunks)
                   ──► 2. turn each section into clean tokens                    (tokenize)
                   ──► 3. pre-compute word statistics                            (BM25Retriever.__post_init__)

customer question  ──► 4. turn the question into clean tokens                    (tokenize)
                   ──► 5. score every section against the question               (BM25Retriever.search)
                   ──► 6. return the best 3                                      → the policy agent's prompt
```

| Part | Section |
|---|---|
| `normalize_arabic` | 1 |
| `_strip_prefix` | 2 |
| `tokenize`, `STOPWORDS` | 3 |
| `_SECTION`, `load_policy_chunks` | 4 |
| BM25 ideas: rarity, repetition, length | 5 |
| A score calculated by hand | 6 |
| `search` | 7 |
| Tuning `k1` and `b` | 8 |
| What it can't do | 9 |
| Exercises | 10 |

## 0. Setup

In [1]:
import math
import sys
from collections import Counter
from pathlib import Path

# Make `cc_copilot` importable whether you start Jupyter from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from cc_copilot.rag.arabic_text import STOPWORDS, _strip_prefix, normalize_arabic, tokenize
from cc_copilot.rag.retriever import _SECTION, BM25Retriever, PolicyChunk, load_policy_chunks

POLICY_DIR = ROOT / "data" / "policies"
print("Policy files:", sorted(p.name for p in POLICY_DIR.glob("*.md")))

Policy files: ['escalation.md', 'identity_verification.md', 'refunds.md']


## 1. `normalize_arabic`: one spelling for each word

People write the same Arabic word in different ways. A customer types `استرجاع` while the policy says
`إسترجاع`; one person writes `مكتبة`, another `مكتبه`. To a computer these are **different strings**, so they
would never match.

`normalize_arabic` makes them identical by:
- removing diacritics (tashkeel: ـَ ـُ ـِ ـّ ـْ …) and the tatweel stretch character `ـ`
- turning every alef form `أ إ آ ٱ` into a plain `ا`
- `ى` → `ي`, `ة` → `ه`, `ؤ` → `و`, `ئ` → `ي`
- lowercasing English letters

In [2]:
examples = ["إِسْتِرْجَاع", "استرجاع", "مكتبة", "مكتبه", "على", "علي", "مســـؤول", "Refund"]
for word in examples:
    print(f"{word:<12} → {normalize_arabic(word)}")

إِسْتِرْجَاع → استرجاع
استرجاع      → استرجاع
مكتبة        → مكتبه
مكتبه        → مكتبه
على          → علي
علي          → علي
مســـؤول     → مسوول
Refund       → refund


Pairs that looked different now produce **identical** output. The cost is that some different words also become
identical (`على` "on" and `علي` "Ali" are now the same). For search that trade-off is almost always worth it.

The same function runs on **both** the policy text and the customer's question. That's the key idea of this
whole file: it doesn't matter if the cleaned form isn't "correct Arabic", only that both sides get cleaned the
same way.

## 2. `_strip_prefix`: removing attached letters

Arabic attaches small words to the front of the next word: `ال` (the), `و` (and), `ب` (with/by), `ل` (for),
`ف` (so), `ك` (like). So `المبلغ`, `والمبلغ` and `بالمبلغ` all contain `مبلغ` but wouldn't match each other.

`_strip_prefix` removes:
1. a two- or three-letter prefix: `وال بال فال كال لل ال`, but only if at least **3 letters** remain;
2. otherwise a single letter `و ب ل ف ك`, but only if the word has at least **5 letters**.

The length rules exist because short words often *start* with those letters as part of the word itself
(`ولد` "boy" is not `و` + `لد`).

In [3]:
words = ["المبلغ", "والمبلغ", "بالتعويض", "للمشرف", "بتعويض", "ولد", "بنك", "كتاب"]
print(f"{'word':<10} {'normalized':<10} → stripped")
for word in words:
    n = normalize_arabic(word)
    print(f"{word:<10} {n:<10} → {_strip_prefix(n)}")

word       normalized → stripped
المبلغ     المبلغ     → مبلغ
والمبلغ    والمبلغ    → مبلغ
بالتعويض   بالتعويض   → تعويض
للمشرف     للمشرف     → مشرف
بتعويض     بتعويض     → تعويض
ولد        ولد        → ولد
بنك        بنك        → بنك
كتاب       كتاب       → كتاب


### A real weakness this reveals

Rule 2 can't tell an attached letter from a letter that belongs to the word:

In [4]:
for word in ["البطاقة", "بطاقة", "فلوسي", "فاتورة"]:
    n = normalize_arabic(word)
    print(f"{word:<8} → {_strip_prefix(n):<6}")

البطاقة  → بطاقه 
بطاقة    → طاقه  
فلوسي    → لوسي  
فاتورة   → اتوره 


- `البطاقة` (the card) → `بطاقه` ✔ (rule 1 removes `ال` and stops)
- `بطاقة` (card) → `طاقه` ✘ which means "energy": rule 2 thinks the `ب` is attached
- `فلوسي` (my money) → `لوسي` ✘

So a customer who writes `بطاقة` **without** `ال` won't match a policy that says `البطاقة`. You'll see this
cause a missed search in section 9. Real Arabic search systems use proper morphological analysers (or embeddings)
for exactly this reason. Fixing it is exercise 1.

## 3. `tokenize`: from a sentence to a list of search words

`tokenize` puts it all together:
1. `normalize_arabic` the whole text
2. split it into words with the regex `\w+` (letters and digits; punctuation is dropped)
3. `_strip_prefix` each word
4. drop **stopwords** (very common words like `في`, `من`, `the`, `and` that appear everywhere and say nothing
   about the topic) and single-letter leftovers

In [5]:
print(sorted(STOPWORDS))

['a', 'an', 'and', 'are', 'be', 'can', 'for', 'i', 'in', 'is', 'it', 'my', 'of', 'on', 'or', 'the', 'to', 'with', 'الي', 'ان', 'او', 'تم', 'ذلك', 'علي', 'عن', 'في', 'قد', 'كل', 'لا', 'ما', 'مع', 'من', 'هذا', 'هذه', 'هل', 'و']


In [6]:
def explain_tokenize(text: str) -> None:
    import re

    print(f"input: {text!r}\n")
    print(f"{'word':<12} {'normalized':<12} {'stripped':<10} result")
    for word in re.findall(r"\w+", text):
        normalized = normalize_arabic(word)
        stripped = _strip_prefix(normalized)
        if stripped in STOPWORDS:
            result = "dropped (stopword)"
        elif len(stripped) <= 1:
            result = "dropped (too short)"
        else:
            result = "KEEP"
        print(f"{word:<12} {normalized:<12} {stripped:<10} {result}")
    print("\ntokenize() returns:", tokenize(text))


explain_tokenize("أبي أسترجع المبلغ في البطاقة، هل هذا ممكن؟")

input: 'أبي أسترجع المبلغ في البطاقة، هل هذا ممكن؟'

word         normalized   stripped   result
أبي          ابي          ابي        KEEP
أسترجع       استرجع       استرجع     KEEP
المبلغ       المبلغ       مبلغ       KEEP
في           في           في         dropped (stopword)
البطاقة      البطاقه      بطاقه      KEEP
هل           هل           هل         dropped (stopword)
هذا          هذا          هذا        dropped (stopword)
ممكن         ممكن         ممكن       KEEP

tokenize() returns: ['ابي', 'استرجع', 'مبلغ', 'بطاقه', 'ممكن']


Note `أبي` → `ابي`: dialect words like `أبي` / `ابغى` ("I want") are kept, since they're not in the stopword
list. They don't hurt much, because they rarely appear in policy text, so they simply match nothing.

## 4. `load_policy_chunks`: splitting policy files into sections

A whole policy file is too big and too mixed to be useful, so each `## <ID> <title>` heading starts a new
**chunk**. Chunks are the unit the retriever returns, and the ID lets the agent cite the exact rule.

The heading regex: `^##\s+(?P<id>[A-Z]+-\d+)\s+(?P<title>.+)$` (with `re.MULTILINE`)

| Piece | Meaning |
|---|---|
| `^` … `$` | start and end of a **line** (that's what `MULTILINE` does) |
| `##\s+` | two hashes, then spaces |
| `(?P<id>[A-Z]+-\d+)` | a named group `id`: capitals, a dash, digits → `REF-1` |
| `\s+(?P<title>.+)` | spaces, then the rest of the line as the named group `title` |

In [7]:
sample = '''# Refunds

## REF-1 Refund eligibility
Within 7 days if unused.

## REF-2 Refund timelines
Card refunds take 5 to 10 business days.
'''

sections = list(_SECTION.finditer(sample))
for i, m in enumerate(sections):
    end = sections[i + 1].start() if i + 1 < len(sections) else len(sample)
    body = sample[m.end() : end].strip()
    print(f"id={m['id']!r}  title={m['title']!r}")
    print(f"  heading ends at {m.end()}, body runs until {end} (next heading or end of file)")
    print(f"  body={body!r}\n")

id='REF-1'  title='Refund eligibility'
  heading ends at 38, body runs until 65 (next heading or end of file)
  body='Within 7 days if unused.'

id='REF-2'  title='Refund timelines'
  heading ends at 90, body runs until 132 (next heading or end of file)
  body='Card refunds take 5 to 10 business days.'



Each section's body is everything **between the end of its heading and the start of the next heading**.
The last section runs to the end of the file. The `# Refunds` line has only one `#`, so it isn't a section.

Now on the real policy files:

In [8]:
chunks = load_policy_chunks(POLICY_DIR)
for c in chunks:
    print(f"{c.policy_id:<6} {c.source:<26} {c.title}")
print(f"\n{len(chunks)} chunks")

ESC-1  escalation.md              Requests for a supervisor / طلب التحدث مع المشرف
ESC-2  escalation.md              Repeat complaints / الشكاوى المتكررة
ESC-3  escalation.md              Tone and courtesy / الأسلوب واللباقة
IDV-1  identity_verification.md   Verifying the caller / التحقق من هوية المتصل
IDV-2  identity_verification.md   Never read back personal data / عدم ترديد البيانات الشخصية
IDV-3  identity_verification.md   Document photos / صور المستندات
REF-1  refunds.md                 Refund eligibility / أهلية استرجاع المبلغ
REF-2  refunds.md                 Refund timelines / مدة استرجاع المبلغ
REF-3  refunds.md                 Goodwill compensation / التعويض

9 chunks


In [9]:
# One chunk in full
print(chunks[0])

PolicyChunk(policy_id='ESC-1', title='Requests for a supervisor / طلب التحدث مع المشرف', text='If a customer asks for a supervisor or manager, the agent must acknowledge the request and transfer or\nschedule a callback within 24 hours. The agent must not ignore or refuse the request.\nإذا طلب العميل التحدث مع المشرف أو المدير، يجب على الموظف الإقرار بالطلب وتحويله أو جدولة اتصال خلال ٢٤ ساعة.', source='escalation.md')


## 5. BM25: the three ideas behind the score

BM25 is the classic search-engine scoring formula (it's been the default in Elasticsearch for years). For every
query word that appears in a chunk, it adds points based on three ideas:

1. **Rarity (IDF).** A word that appears in few chunks is a strong signal; a word in every chunk is useless.
2. **Repetition, with diminishing returns (TF + `k1`).** A chunk that mentions the word more often is more
   relevant, but the 10th mention adds far less than the 1st.
3. **Length (`b`).** A long chunk mentions many words just by being long, so its matches count a little less.

When the retriever is created, `__post_init__` pre-computes everything that doesn't depend on the question:
the tokens of every chunk, the IDF of every word, and the average chunk length.

In [10]:
retriever = BM25Retriever(chunks)

print("tokens per chunk:")
for c, tokens in zip(retriever.chunks, retriever._doc_tokens):
    print(f"  {c.policy_id:<6} {len(tokens):>3} tokens   first few: {tokens[:6]}")
print(f"\naverage length: {retriever._avg_len:.1f} tokens")

tokens per chunk:
  ESC-1   42 tokens   first few: ['requests', 'supervisor', 'طلب', 'تحدث', 'مشرف', 'if']
  ESC-2   27 tokens   first few: ['repeat', 'complaints', 'شكاوي', 'متكرره', 'complaint', 'about']
  ESC-3   42 tokens   first few: ['tone', 'courtesy', 'اسلوب', 'لباقه', 'agents', 'must']
  IDV-1   36 tokens   first few: ['verifying', 'caller', 'تحقق', 'هويه', 'متصل', 'before']
  IDV-2   50 tokens   first few: ['never', 'read', 'back', 'personal', 'data', 'عدم']
  IDV-3   37 tokens   first few: ['document', 'photos', 'صور', 'مستندات', 'if', 'customer']
  REF-1   50 tokens   first few: ['refund', 'eligibility', 'اهليه', 'استرجاع', 'مبلغ', 'customer']
  REF-2   38 tokens   first few: ['refund', 'timelines', 'مده', 'استرجاع', 'مبلغ', 'agents']
  REF-3   45 tokens   first few: ['goodwill', 'compensation', 'تعويض', 'frontline', 'agents', 'may']

average length: 40.8 tokens


### 5a. Rarity: IDF

**df** (document frequency) = in how many chunks a word appears. Then

$$\text{IDF} = \ln\left(1 + \frac{N - df + 0.5}{df + 0.5}\right)$$

where $N$ is the number of chunks (9). A word in 1 chunk gets a high IDF; a word in most chunks gets a low one.

In [11]:
df = Counter(t for doc in retriever._doc_tokens for t in set(doc))
N = len(retriever.chunks)

print(f"{'word':<10} {'in chunks':>9} {'IDF':>6}")
for word in ["مدير", "مشرف", "مبلغ", "يجب", "موظف", "agent", "customer"]:
    print(f"{word:<10} {df[word]:>9} {retriever._idf.get(word, 0):>6.2f}")

word       in chunks    IDF
مدير               1   1.90
مشرف               2   1.39
مبلغ               2   1.39
يجب                4   0.80
موظف               5   0.60
agent              2   1.39
customer           7   0.29


`مدير` (manager) appears in only one chunk, so finding it is very telling. `customer` is in nearly every chunk,
so it barely helps choose between them.

### 5b. Repetition with diminishing returns: `k1`

For a word that appears `freq` times, BM25 uses $\frac{freq \cdot (k_1 + 1)}{freq + k_1}$ (ignoring length for
now). With `k1 = 1.5`:

In [12]:
k1 = retriever.k1
print(f"{'freq':>4}  {'TF part':>7}  gain from this extra mention")
previous = 0
for freq in range(1, 11):
    tf_part = freq * (k1 + 1) / (freq + k1)
    print(f"{freq:>4}  {tf_part:>7.2f}  +{tf_part - previous:.2f}")
    previous = tf_part
print(f"\nIt can never go above k1 + 1 = {k1 + 1}")

freq  TF part  gain from this extra mention
   1     1.00  +1.00
   2     1.43  +0.43
   3     1.67  +0.24
   4     1.82  +0.15
   5     1.92  +0.10
   6     2.00  +0.08
   7     2.06  +0.06
   8     2.11  +0.05
   9     2.14  +0.04
  10     2.17  +0.03

It can never go above k1 + 1 = 2.5


The first mention is worth 1.00, the second adds 0.43, and by the tenth almost nothing. This stops a chunk from
winning just by repeating one word over and over.

### 5c. Length: `b`

The length factor is $1 - b + b \cdot \frac{\text{chunk length}}{\text{average length}}$. It replaces `k1` with
`k1 × norm` in the formula above:
- a chunk of **average** length gets 1.0 (no change)
- a **longer** chunk gets more than 1, which shrinks its score
- a **shorter** chunk gets less than 1, which boosts it

In [13]:
b = retriever.b
print(f"{'chunk':<6} {'length':>6} {'norm':>6}")
for c, tokens in zip(retriever.chunks, retriever._doc_tokens):
    norm = 1 - b + b * len(tokens) / retriever._avg_len
    print(f"{c.policy_id:<6} {len(tokens):>6} {norm:>6.2f}")

chunk  length   norm
ESC-1      42   1.02
ESC-2      27   0.75
ESC-3      42   1.02
IDV-1      36   0.91
IDV-2      50   1.17
IDV-3      37   0.93
REF-1      50   1.17
REF-2      38   0.95
REF-3      45   1.08


## 6. A score calculated by hand

For each chunk, BM25 adds up one term per query word found in it:

$$\text{score} = \sum_{\text{word in query}} \text{IDF}(word) \cdot \frac{freq \cdot (k_1 + 1)}{freq + k_1 \cdot norm}$$

This cell does the calculation step by step, then checks it against the real `search()`:

In [ ]:
def explain_score(query: str) -> None:
    q_tokens = tokenize(query)
    print(f"query: {query!r}\ntokens: {q_tokens}\n")
    for c, tokens in zip(retriever.chunks, retriever._doc_tokens):
        tf = Counter(tokens)
        norm = 1 - retriever.b + retriever.b * len(tokens) / retriever._avg_len
        total, parts = 0.0, []
        for t in q_tokens:
            if t not in tf:
                continue
            idf = retriever._idf[t]
            term = idf * tf[t] * (retriever.k1 + 1) / (tf[t] + retriever.k1 * norm)
            parts.append(f"{t}: idf {idf:.2f} × freq {tf[t]} → {term:.2f}")
            total += term
        if parts:
            print(f"{c.policy_id}  (norm {norm:.2f})")
            for p in parts:
                print("    ", p)
            print(f"     total = {total:.2f}\n")

    print("search() says:", [(h.chunk.policy_id, round(h.score, 2)) for h in retriever.search(query)])

explain_score("أبي أسترجع المبلغ")

query: 'أبي أسترجع المبلغ'
tokens: ['ابي', 'استرجع', 'مبلغ']

REF-1  (norm 1.17)
     مبلغ: idf 1.39 × freq 3 → 2.19
     total = 2.19

REF-2  (norm 0.95)
     مبلغ: idf 1.39 × freq 2 → 2.02
     total = 2.02

search() says: [('REF-1', 2.19), ('REF-2', 2.02)]


The hand calculation and `search()` agree. Chunks that share **no** words with the question score 0 and aren't
shown at all.

Look closely: only `مبلغ` scored. The customer wrote `أسترجع` (the verb, "I get back"), but the policy says
`استرجاع` (the noun, "refund"). They come from the same root, yet as strings they're different, so they don't
match. REF-1 still won thanks to `مبلغ`, but this is the same weakness as section 2: light prefix stripping
doesn't understand Arabic word forms.

## 7. `search`: questions in different dialects

`search(query, top_k=3)`:
1. tokenizes the question,
2. scores every chunk (section 6),
3. **drops chunks scoring 0**, so an unrelated question returns an empty list instead of 3 random policies,
4. sorts from highest to lowest score and keeps the first `top_k`.

Step 3 matters for the policy agent: an empty result tells it "no policy applies", instead of feeding it
irrelevant rules that it might wrongly apply.

In [15]:
def show_search(query: str, top_k: int = 3) -> None:
    hits = retriever.search(query, top_k=top_k)
    result = ", ".join(f"{h.chunk.policy_id} ({h.score:.2f})" for h in hits) or "nothing"
    print(f"{query:<36} → {result}")


show_search("أبي أسترجع المبلغ")              # Najdi: I want a refund
show_search("ابغى اكلم المدير")               # Hijazi: I want to talk to the manager
show_search("الموظف وعدني بتعويض ٢٠٠ ريال")   # the agent promised me 200 SAR compensation
show_search("can you read my IBAN back to me")
show_search("weather tomorrow in Abha")      # unrelated → nothing

أبي أسترجع المبلغ                    → REF-1 (2.19), REF-2 (2.02)
ابغى اكلم المدير                     → ESC-1 (1.87)
الموظف وعدني بتعويض ٢٠٠ ريال         → REF-3 (3.19), REF-2 (0.62), ESC-1 (0.59)
can you read my IBAN back to me      → IDV-2 (5.51), REF-1 (1.26)
weather tomorrow in Abha             → nothing


`top_k` controls how many sections the agent receives:

In [16]:
show_search("refund", top_k=1)
show_search("refund", top_k=5)

refund                               → REF-2 (2.02)
refund                               → REF-2 (2.02), REF-1 (1.85)


## 8. Tuning `k1` and `b`

`k1 = 1.5` and `b = 0.75` are the usual textbook defaults. Here's how changing them shifts the scores for the same
question. With `b = 0` length is ignored; with `b = 1` length matters fully:

In [17]:
query = "refund card days"
for k1, b in [(1.5, 0.75), (1.5, 0.0), (1.5, 1.0), (0.5, 0.75), (3.0, 0.75)]:
    r = BM25Retriever(chunks, k1=k1, b=b)
    hits = ", ".join(f"{h.chunk.policy_id} {h.score:.2f}" for h in r.search(query))
    print(f"k1={k1:<4} b={b:<5} → {hits}")

k1=1.5  b=0.75  → REF-2 5.48, REF-1 3.69, IDV-3 1.45
k1=1.5  b=0.0   → REF-2 5.35, REF-1 3.96, IDV-3 1.39
k1=1.5  b=1.0   → REF-2 5.53, REF-1 3.61, IDV-3 1.47
k1=0.5  b=0.75  → REF-2 4.77, REF-1 3.22, IDV-3 1.42
k1=3.0  b=0.75  → REF-2 6.02, REF-1 4.03, IDV-3 1.46


With only 9 small chunks the ranking barely changes; the defaults are fine. Tuning only becomes worthwhile once
you have **many** policies **and** a labelled set of questions to measure with (exercise 3), otherwise you're
tuning to your gut feeling.

## 9. What it can't do (and why embeddings come next)

BM25 matches **words, not meaning**. These all fail or go wrong:

In [18]:
show_search("رجعوا لي فلوسي")        # "give me my money back": no shared words with "استرجاع المبلغ"
show_search("بطاقة")                # becomes طاقه (energy), see section 2, so it misses REF-2
show_search("البطاقة")              # same word with ال → works
show_search("abi aklm el modeer")   # Arabizi: the policies have no Arabizi text
show_search("I want my money back") # matches only because both chunks contain the word "back"

رجعوا لي فلوسي                       → nothing
بطاقة                                → nothing
البطاقة                              → REF-2 (1.96)
abi aklm el modeer                   → nothing
I want my money back                 → IDV-2 (1.26), REF-1 (1.26)


- **Synonyms and dialect vocabulary**: `فلوسي` and `المبلغ` mean the same thing, but share no letters.
- **Stemming mistakes**: `بطاقة` vs `البطاقة` (section 2).
- **Arabizi**: Latin-letter Arabic only matches if it was written exactly the same way.
- **Accidental matches**: "back" in "money back" and in "read back personal data" are unrelated meanings.

**Embeddings** fix most of these: they turn text into vectors where sentences with similar *meaning* are close
together, whatever the words. That's Phase 5 of the roadmap: add embedding search, combine it with BM25
("hybrid search"), and **measure** which works better on your labelled questions. BM25 stays useful for exact
terms like `IBAN`, `REF-2` or `OTP`, which embeddings sometimes blur.

## 10. Exercises

1. **Fix the `بطاقة` bug.** Change `_strip_prefix` so `بطاقة` and `البطاقة` give the same token. One idea:
   only strip a single letter if what remains appears in a list of known policy words. Re-run the tests
   (`uv run pytest`) to check you didn't break anything, and add a test for this case.
2. **Add a policy.** Add a section such as `## ROAM-1 Roaming charges / رسوم التجوال` to a policy file, re-run
   section 4, and check that a roaming question finds it.
3. **Measure, don't guess.** Write 10+ questions labelled with the policy that should come first (use the cell
   below as a start), and compute how often the right policy is ranked first (**hit@1**) and in the top 3
   (**hit@3**). This is the baseline you'll compare embeddings against in Phase 5.
4. **Add stopwords.** Look at the tokens in section 5. Are there dialect words (`ابي`, `ابغي`, `وين`) that should be
   stopwords? Add them and see whether any search results change.

In [19]:
labelled = [
    ("أبي أسترجع المبلغ", "REF-1"),
    ("متى توصل فلوسي للبطاقة", "REF-2"),
    ("الموظف وعدني بتعويض ٢٠٠ ريال", "REF-3"),
    ("ابغى اكلم المدير", "ESC-1"),
    ("ثالث مرة اشتكي على نفس المشكلة", "ESC-2"),
    ("can you read my IBAN back to me", "IDV-2"),
    ("أرسلت صورة الهوية، وش تسوون فيها؟", "IDV-3"),
]

hit1 = hit3 = 0
for question, expected in labelled:
    ranked = [h.chunk.policy_id for h in retriever.search(question, top_k=3)]
    hit1 += ranked[:1] == [expected]
    hit3 += expected in ranked
    mark = "✔" if ranked[:1] == [expected] else ("~" if expected in ranked else "✘")
    print(f"{mark} {question:<36} expected {expected:<6} got {ranked}")

print(f"\nhit@1 = {hit1 / len(labelled):.0%}   hit@3 = {hit3 / len(labelled):.0%}")

✔ أبي أسترجع المبلغ                    expected REF-1  got ['REF-1', 'REF-2']
✔ متى توصل فلوسي للبطاقة               expected REF-2  got ['REF-2']
✔ الموظف وعدني بتعويض ٢٠٠ ريال         expected REF-3  got ['REF-3', 'REF-2', 'ESC-1']
✔ ابغى اكلم المدير                     expected ESC-1  got ['ESC-1']
✔ ثالث مرة اشتكي على نفس المشكلة       expected ESC-2  got ['ESC-2', 'REF-2']
✔ can you read my IBAN back to me      expected IDV-2  got ['IDV-2', 'REF-1']
✔ أرسلت صورة الهوية، وش تسوون فيها؟    expected IDV-3  got ['IDV-3', 'IDV-1', 'IDV-2']

hit@1 = 100%   hit@3 = 100%


✔ = right policy ranked first, ~ = in the top 3 but not first, ✘ = missed.

These starter questions all pass because they were written using words from the policies. **Real customers won't
be that kind.** Add the hard questions from section 9, plus your own in different dialects and in Arabizi, and
watch the score drop. Each ✘ is worth investigating with `explain_tokenize` and `explain_score`: is it a stemming
problem, a missing word, or a synonym? A low score here is useful, not embarrassing: it's the number embeddings
must beat in Phase 5.